In [ ]:
from pathlib import Path
import glob
import os
import re

import numpy as np
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import pandas as pd

In [ ]:
# Input data paths
country_thresholds_path = "../data/country_thresholds.csv"
wb_income_classification_path = "../data/wb_income_classification.csv"
results_dir = "/data/tc-grid/open-gira/results"

# Output paths
plot_dir = Path("../plots")

# Do not extrapolate beyond the available events
tc_model_n_years = {
    "CHAZ": 1000,
    "emanuel": 200,
    "IRIS": 1000,
    "STORM": 1000,
}

# Mapping from open-gira STORM_SET to display name
atmospheres_to_scenario = {
    "STORM_constant": "2000 STORM",
    "STORM_CNRM-CM6-1-HR": "2050 STORM RCP8.5 CNRM",
    "STORM_CMCC-CM2-VHR4": "2050 STORM RCP8.5 CMCC",
    "STORM_EC-Earth3P-HR": "2050 STORM RCP8.5 EC-Earth",
    "STORM_HadGEM3-GC31-HM": "2050 STORM RCP8.5 HadGEM3",
    "IRIS_PRESENT": "2000 IRIS",
    "IRIS_SSP5-2050": "2050 IRIS SSP5-8.5",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2010": "2010 CHAZ UKESM1",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2010": "2010 CHAZ CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2010": "2010 CHAZ CNRM",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2010": "2010 CHAZ EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2010": "2010 CHAZ MIROC6",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2050": "2050 CHAZ SSP5-8.5 UKESM1",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2050": "2050 CHAZ SSP5-8.5 CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2050": "2050 CHAZ SSP5-8.5 CNRM",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2050": "2050 CHAZ SSP5-8.5 EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2050": "2050 CHAZ SSP5-8.5 MIROC6",
    "emanuel_ssp-585_gcm-cesm2_epoch-2005": "2005 Emanuel CESM2",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2005": "2005 Emanuel CNRM6",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2005": "2005 Emanuel EC-Earth6",
    #"emanuel_ssp-585_gcm-fgoals_epoch-2005": "2005 Emanuel FGOALS",
    #"emanuel_ssp-585_gcm-ipsl6_epoch-2005": "2005 Emanuel IPSL6",
    "emanuel_ssp-585_gcm-miroc6_epoch-2005": "2005 Emanuel MIROC6",
    #"emanuel_ssp-585_gcm-mpi6_epoch-2005": "2005 Emanuel MPI6",
    #"emanuel_ssp-585_gcm-mri6_epoch-2005": "2005 Emanuel MRI6",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2005": "2005 Emanuel UKMO6",
    "emanuel_ssp-585_gcm-cesm2_epoch-2050": "2050 Emanuel CESM2",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2050": "2050 Emanuel CNRM6",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2050": "2050 Emanuel EC-Earth6",
    #"emanuel_ssp-585_gcm-fgoals_epoch-2050": "2050 Emanuel FGOALS",
    #"emanuel_ssp-585_gcm-ipsl6_epoch-2050": "2050 Emanuel IPSL6",
    "emanuel_ssp-585_gcm-miroc6_epoch-2050": "2050 Emanuel MIROC6",
    #"emanuel_ssp-585_gcm-mpi6_epoch-2050": "2050 Emanuel MPI6",
    #"emanuel_ssp-585_gcm-mri6_epoch-2050": "2050 Emanuel MRI6",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2050": "2050 Emanuel UKMO6",
}
# Specify group members (GCM variants) of track sets
gcm_groups = {
    "IRIS baseline": ["2000 IRIS"],
    "IRIS SSP5-8.5 2050": ["2050 IRIS SSP5-8.5"],
    "STORM baseline": ["2000 STORM"],
    "STORM RCP8.5 2050": [
        "2050 STORM RCP8.5 CNRM",
        "2050 STORM RCP8.5 CMCC",
        "2050 STORM RCP8.5 EC-Earth",
        "2050 STORM RCP8.5 HadGEM3"
    ],
    "CHAZ baseline": [
        "2010 CHAZ UKESM1",
        "2010 CHAZ CESM2",
        "2010 CHAZ CNRM",
        "2010 CHAZ EC-Earth3",
        "2010 CHAZ MIROC6",
    ],
    "CHAZ SSP5-8.5 2050": [
        "2050 CHAZ SSP5-8.5 UKESM1",
        "2050 CHAZ SSP5-8.5 CESM2",
        "2050 CHAZ SSP5-8.5 CNRM",
        "2050 CHAZ SSP5-8.5 EC-Earth3",
        "2050 CHAZ SSP5-8.5 MIROC6",
    ],   
    "MIT baseline": [
        "2005 Emanuel CESM2",
        "2005 Emanuel CNRM6",
        "2005 Emanuel EC-Earth6",
        #"2005 Emanuel FGOALS",
        #"2005 Emanuel IPSL6",
        "2005 Emanuel MIROC6",
        #"2005 Emanuel MPI6",
        #"2005 Emanuel MRI6",
        "2005 Emanuel UKMO6",
    ],
    "MIT SSP5-8.5 2050": [
        "2050 Emanuel CESM2",
        "2050 Emanuel CNRM6",
        "2050 Emanuel EC-Earth6",
        #"2050 Emanuel FGOALS",
        #"2050 Emanuel IPSL6",
        "2050 Emanuel MIROC6",
        #"2050 Emanuel MPI6",
        #"2050 Emanuel MRI6",
        "2050 Emanuel UKMO6",
    ],
}
group_colours = {
    "IRIS": "lightgreen",
    "STORM": "plum",
    "CHAZ": "pink",
    "MIT": "lightblue",
}

In [ ]:
# Assemble optimum country to threshold mapping
# Calibrated thresholds, ms-1
per_country_calibrated_thresholds = pd.read_csv(country_thresholds_path).rename(columns={"iso_a3": "ISO_A3"}).set_index("ISO_A3")
# Thresholds based on income, ms-1
income_threshold_map = {"L": "30.0", "LM": "30.0", "UM": "30.0", "H": "32.5"}
thresholds = pd.read_csv(wb_income_classification_path, comment="#").set_index("ISO_A3")
thresholds["threshold_ms-1"] = thresholds.INCOME_LEVEL.map(income_threshold_map)
thresholds.loc[per_country_calibrated_thresholds.index, "threshold_ms-1"] = per_country_calibrated_thresholds["threshold_ms-1"].astype(str)

#rps = np.array([i * base for base in (1, 10, 100) for i in range(1, 10)])
rps = np.logspace(0, 3, 40)
baseline = {}
future = {}
for atmosphere, scenario in atmospheres_to_scenario.items():
    
    print(atmosphere, end="\r")
    paths = glob.glob(os.path.join(results_dir, f"power/by_country/*/disruption/{atmosphere}/pop_affected_by_event.pq"))
    data = {path.split("/")[-4]: pd.read_parquet(path,) for path in paths}
    calibrated_data = {}
    for iso_a3, df in data.items():
        try:
            threshold = thresholds.loc[iso_a3, "threshold_ms-1"]
            calibrated_data[iso_a3] = df.loc[:, str(threshold)]
        except KeyError:
            pass
            #print(f"{iso_a3} missing...")
            
    concat = pd.concat(calibrated_data.values())
    concat = pd.DataFrame(concat).rename(columns={0: "pop_affected"})
    
    global_sum = concat.groupby(concat.index).sum()
    global_sum = global_sum.reset_index()
    if atmosphere.startswith("STORM") or atmosphere.startswith("IRIS"):
        global_sum["year"] = global_sum.event_id.apply(lambda s: int(s.split("_")[2]))
    elif atmosphere.startswith("CHAZ") or atmosphere.startswith("emanuel"):
        global_sum["year"] = global_sum.event_id.apply(lambda s: int(re.search(r"Y(\d{4})", s).groups()[0]))
        
    annual_max = global_sum.drop(columns=["event_id"]).groupby("year").max()
    annual_sum = global_sum.drop(columns=["event_id"]).groupby("year").sum()
    
    df = annual_sum.sort_values("pop_affected")
    df = df.reset_index()
    
    # For any atmospheres with more than 1 millenium, drop extra years
    df = df[(df.year >= 0) & (df.year < 1000)]

    # Label with plotting position
    n_years = df.year.max() - df.year.min() + 1
    year_range = range(n_years - 1, -1, -1)
    df["rank"] = year_range
    df["return_period_years"] = (n_years + 1) / df["rank"]

    # Interpolate to common RP grid
    df = df.drop(columns=["year", "rank"])
    df = df[np.isfinite(df.return_period_years)]
    df["log_rp"] = np.log(df.return_period_years)
    target = pd.DataFrame({"return_period_years": rps, "log_rp": np.log(rps), "pop_affected": np.nan})
    df = pd.concat([df, target]).sort_values("log_rp").set_index("log_rp")
    df = df.interpolate(method="index")
    df = df.loc[np.log(rps)].reset_index(drop=True)

    years_of_events = tc_model_n_years[atmosphere.split("_")[0]]
    df.loc[df.return_period_years > years_of_events, "pop_affected"] = np.nan

    for group_name, members in gcm_groups.items():
        if scenario in members:
            break

    family, *other = group_name.split()
    if other[0] == "baseline":
        baseline[(family, scenario)] = df
    else:
        future[(family, scenario)] = df

In [ ]:
f = plt.figure(figsize=(10, 4))
outer = GridSpec(1, 2, width_ratios=[2, 1], wspace=0.27, figure=f)  # Gap between (1,2) and 3
inner = outer[0].subgridspec(1, 2, wspace=0.05)  # Gap between 1 and 2

baseline_ax = f.add_subplot(inner[0])
future_ax = f.add_subplot(inner[1])
ratio_ax = f.add_subplot(outer[1])

alpha = 0.8
labelsize = 10
iqr_ls = "-"
iqr_lw = 0.5
iqr_alpha = 0.2

for ax, dataset in ((baseline_ax, baseline), (future_ax, future)):
    q25 = pd.concat(dataset).groupby("return_period_years").agg(lambda x: np.nanquantile(x, 0.25)).reset_index()
    q50 = pd.concat(dataset).groupby("return_period_years").agg(np.nanmedian).reset_index()
    q75 = pd.concat(dataset).groupby("return_period_years").agg(lambda x: np.nanquantile(x, 0.75)).reset_index()
    
    for (family, scenario), df in dataset.items():    
        ax.plot(
            df["return_period_years"],
            df["pop_affected"],
            label=scenario,
            alpha=alpha,
            color=group_colours[family],
        )

    ax.fill_between(q25["return_period_years"], q25["pop_affected"], q75["pop_affected"], color="grey", alpha=iqr_alpha)
    ax.plot(q25["return_period_years"], q25["pop_affected"], color="black", ls=iqr_ls, lw=iqr_lw, alpha=0.5)
    ax.plot(q50["return_period_years"], q50["pop_affected"], color="black", lw=1, ls="--")
    ax.plot(q75["return_period_years"], q75["pop_affected"], color="black", ls=iqr_ls, lw=iqr_lw, alpha=0.5)

# Invisible handle used purely as a spacer/header placeholder
blank = Line2D([0], [0], color="none")
handles = [
    blank,
    *[Line2D([0], [0], color=c, lw=2, alpha=alpha) for c in group_colours.values()],
    blank,
    Line2D([0], [0], color="black", lw=2, alpha=1, ls="--"),
    mpatches.Patch(color='grey', alpha=0.5, edgecolor="black"),
]
labels = [r"$\bf{Model}$", *group_colours.keys(), "", "Median", "IQR"]
future_ax.legend(
    handles, labels,
    loc="lower right", bbox_to_anchor=(1, 0),
    frameon=False, handlelength=2.0, labelspacing=0.6,
    fontsize=9,
)
ymax = 1.2 * max([pd.concat(baseline).pop_affected.max(), pd.concat(future).pop_affected.max()])
for ax in (baseline_ax, future_ax):
    ax.set_xlim(2, 500)
    ax.set_ylim(8E6, ymax)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.grid(alpha=0.2, which="both")

# How does the future vs. baseline ratio change as a function of RP?
agg = np.nanmedian
df = pd.concat(
    [
        pd.concat(baseline).groupby("return_period_years").agg(agg).rename(columns={"pop_affected": "baseline"}),
        pd.concat(future).groupby("return_period_years").agg(agg).rename(columns={"pop_affected": "future"})
    ],
    axis=1
)
df = df.reset_index()
df["ratio"] = df.future / df.baseline

ratio_ax.set_ylabel("Future / baseline", fontsize=labelsize, labelpad=10)
ratio_ax.plot(df["return_period_years"], df["ratio"], c="teal")
ratio_ax.set_xscale("log")
ratio_ax.grid(which="both", alpha=0.15)
ratio_ax.set_xlim(1, 500)
_, ymax = ratio_ax.get_ylim()
ratio_ax.set_ylim(0.97, 1.02 * ymax)

future_ax.set_yticklabels([])
for ax, text in ((baseline_ax, "a) Baseline"), (future_ax, "b) 2050 RCP8.5/SSP5-8.5"), (ratio_ax, "c) Ratio")):
    ax.text(0.04, 0.95, text, fontsize=labelsize, horizontalalignment='left', verticalalignment='top', transform=ax.transAxes)

f.supxlabel("Return period [years]", fontsize=labelsize, y=0.05)
baseline_ax.set_ylabel("Population at risk", fontsize=labelsize, x=0.03)
    
plt.subplots_adjust(left=0.08, right=0.97, bottom=0.19, top=0.95, wspace=0.05)
f.savefig(plot_dir / "global_exceedance_spaghetti.pdf", bbox_inches="tight")

In [ ]:
# What are the q25, 50 and 75 figures for baseline risk?
df = pd.concat(baseline).groupby("return_period_years")
q25 = df.agg(lambda x: np.quantile(x, 0.25)).rename(columns={"pop_affected": "q25"})
q50 = df.agg(np.median).rename(columns={"pop_affected": "q50"})
q75 = df.agg(lambda x: np.quantile(x, 0.75)).rename(columns={"pop_affected": "q75"})
data = pd.concat([q25, q50, q75], axis=1)
print(data.loc[45: 55])
print(data.loc[90: 110])

In [ ]:
rp = 50
data = {}
for epoch, dataset in (("Baseline", baseline), ("Future", future)):
    for (family, model), pop_affected in dataset.items():
        data[(epoch, family, model)] = pop_affected.set_index("return_period_years")
df = pd.concat(data).sort_index()
out = df.loc[:, :, :, rp - 1: rp + 1]
out

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# out has MultiIndex (epoch, family, model) -> pop_affected
epochs = out.index.get_level_values(0).unique().tolist()
families = out.index.get_level_values(1).unique().tolist()

fig, ax = plt.subplots(figsize=(6, 4))

group_width = 0.6
fam_width = group_width / len(families)
rng = np.random.default_rng(0)

for i, epoch in enumerate(epochs):
    epoch_vals = out.loc[epoch]["pop_affected"]

    for j, family in enumerate(families):
        if family not in epoch_vals.index.get_level_values(0):
            continue
        vals = epoch_vals.loc[family].values
        x_center = i + (j - (len(families) - 1) / 2) * fam_width
        x = rng.normal(x_center, fam_width * 0.08, size=len(vals))
        ax.scatter(
            x, vals,
            #color=colors[j % len(colors)],
            color=group_colours[family],
            label=family if i == 0 else None,
            alpha=1, zorder=3,
        )

    # dashed median line across the whole epoch group
    median = epoch_vals.median()
    ax.hlines(
        median,
        i - group_width / 2, i + group_width / 2,
        colors="black", linestyles="--", linewidth=1.5, zorder=2,
    )

ax.grid(alpha=0.2)
ax.set_xticks(range(len(epochs)))
ax.set_xticklabels(epochs)
ax.set_ylim(4E7, 1.6E8)
ax.set_ylabel(f"Population at risk: {1/rp:.2f} AEP")

# Combine auto-generated scatter legend entries with a proxy for the median line
handles, labels = ax.get_legend_handles_labels()
median_handle = Line2D([0], [0], color="black", linestyle="--", linewidth=1.5)
handles.append(median_handle)
labels.append("Median")
ax.legend(handles, labels, title="TC model", bbox_to_anchor=(1.02, 1), loc="upper left")

fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# swap levels: (epoch, family, model) -> (family, epoch, model)
out_swapped = out.swaplevel(0, 1).sort_index(level=0)

families = out_swapped.index.get_level_values(0).unique().tolist()
epochs = out_swapped.index.get_level_values(1).unique().tolist()

fig, ax = plt.subplots(figsize=(6, 4))

group_width = 0.6
epoch_width = group_width / len(epochs)
rng = np.random.default_rng(0)
colours = ["plum", "lightblue"]

for i, family in enumerate(families):
    fam_vals = out_swapped.loc[family]["pop_affected"]

    for j, epoch in enumerate(epochs):
        if epoch not in fam_vals.index.get_level_values(0):
            continue
        vals = fam_vals.loc[epoch].values
        x_center = i + (j - (len(epochs) - 1) / 2) * epoch_width
        x = rng.normal(x_center, epoch_width * 0.08, size=len(vals))
        ax.scatter(
            x, vals,
            color=colours[j % len(colours)],
            label=epoch if i == 0 else None,
            alpha=1, zorder=3,
        )

ax.grid(alpha=0.2)
ax.set_xticks(range(len(families)))
ax.set_xticklabels(families)
ax.set_ylabel("Population affected")
ax.legend(title="Epoch", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()